# Lab 13: Web Search pour Modèles SOTA (MLE-STAR Component)

**Navigation** : [Lab 12 <<](../Day5-DS-Star/Lab12-DS-Star-Workshop.ipynb) | [Index](../../README.md) | [>> Lab 14](Lab14-Ablation-Refinement.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Implémenter la recherche web pour trouver les modèles SOTA
2. Extraire des informations depuis arXiv et autres sources académiques
3. Générer du code initial basé sur les résultats de recherche
4. Intégrer la recherche dans un pipeline ML automatisé

### Prérequis
- Lab 12 (DS-STAR Workshop) complété
- Compréhension des architectures ML
- Configuration multi-provider active

### Durée estimée : 35-45 minutes

## 1. Configuration

In [1]:
import sys
sys.path.insert(0, '..')

import json
import re
import requests
from typing import List, Dict, Optional
from dataclasses import dataclass
from datetime import datetime

from config import get_settings
from utils import LLMClient
print("Imports OK : json, re, requests")

Imports OK : json, re, requests


Chargement des parametres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openrouter


## 2. Data Classes

In [3]:
@dataclass
class SearchResult:
    title: str
    url: str
    snippet: str
    source: str
    date: str = None

@dataclass
class ModelInfo:
    name: str
    paper_url: str
    description: str
    code_url: str = None
    performance: str = None
print("Dataclasses definies : SearchResult, ModelInfo")

Dataclasses definies : SearchResult, ModelInfo


## 3. Web Search Module

In [4]:
class WebSearcher:
    """Recherche web pour trouver des modeles SOTA."""

    def __init__(self):
        self.headers = {'User-Agent': 'Mozilla/5.0 (compatible; DS-Star/1.0)'}

    def search_arxiv(self, query: str, max_results: int = 5) -> List[SearchResult]:
        """Recherche sur arXiv via API."""
        url = f"http://export.arxiv.org/api/query?search_query=all:{query}&max_results={max_results}"
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            results = []
            # Parse Atom feed
            entries = response.text.split('<entry>')[1:]
            for entry in entries:
                title = re.search(r'<title>(.*?)</title>', entry, re.DOTALL)
                link = re.search(r'<id>(.*?)</id>', entry)
                summary = re.search(r'<summary>(.*?)</summary>', entry, re.DOTALL)
                if title and link:
                    results.append(SearchResult(
                        title=title.group(1).strip().replace('\n', ' '),
                        url=link.group(1).strip(),
                        snippet=summary.group(1).strip()[:200] if summary else '',
                        source='arxiv'
                    ))
            return results
        except Exception as e:
            print(f"Erreur arXiv: {e}")
            return []

    def search_papers_with_code(self, task: str) -> List[SearchResult]:
        """Simule une recherche sur Papers With Code (API publique)."""
        # Papers With Code a une API publique mais limitee
        # Pour ce lab, on simule avec des resultats types
        mock_results = [
            SearchResult(
                title="State-of-the-Art Image Classification",
                url="https://paperswithcode.com/sota/image-classification-on-imagenet",
                snippet="Vision Transformers achieve 90.5% top-1 accuracy",
                source="paperswithcode"
            ),
            SearchResult(
                title="Object Detection Benchmarks",
                url="https://paperswithcode.com/sota/object-detection-on-coco",
                snippet="YOLOv9 and DINO lead COCO detection",
                source="paperswithcode"
            )
        ]
        return mock_results
print("Classe WebSearcher definie")

Classe WebSearcher definie


## 4. LLM-Based Model Extractor

In [5]:
class ModelExtractor:
    """Extrait les informations de modeles depuis les resultats de recherche."""

    def __init__(self, llm: LLMClient):
        self.llm = llm

    def extract_models(self, search_results: List[SearchResult], task: str) -> List[ModelInfo]:
        """Utilise le LLM pour extraire les modeles pertinents."""
        context = "\n".join([
            f"- {r.title}: {r.snippet} ({r.url})"
            for r in search_results[:5]
        ])

        prompt = f"""Analyse ces resultats de recherche pour la tache: {task}

RESULTATS:
{context}

Extrait les 2-3 modeles les plus pertinents avec leurs informations.
Format JSON:
[
  {{"name": "...", "paper_url": "...", "description": "...", "code_url": "..."}}
]

JSON:"""

        response = self.llm.generate(prompt, temperature=0.2)

        # Extract JSON
        try:
            match = re.search(r'\[.*\]', response, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                return [ModelInfo(**m) for m in data]
        except:
            pass
        return []
print("Classe ModelExtractor definie")

Classe ModelExtractor definie


## 5. Code Generator from SOTA

In [6]:
class SOTACodeGenerator:
    """Genere du code initial base sur les modeles SOTA trouves."""

    def __init__(self, llm: LLMClient):
        self.llm = llm

    def generate_initial_code(self, task: str, models: List[ModelInfo]) -> str:
        """Genere du code de base utilisant les modeles SOTA."""
        models_desc = "\n".join([
            f"- {m.name}: {m.description}"
            for m in models[:2]
        ])

        prompt = f"""Genere du code Python initial pour cette tache ML.

TACHE: {task}

MODELES SOTA DISPONIBLES:
{models_desc}

Genere un script Python simple et commente qui:
1. Charge les donnees
2. Prepare le modele
3. Entra�ne et evalue

```python
# Ton code ici
```"""

        response = self.llm.generate(prompt, temperature=0.3)
        match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
        return match.group(1).strip() if match else response
print("Classe SOTACodeGenerator definie")

Classe SOTACodeGenerator definie


## 6. MLE-STAR Pipeline Partiel

In [7]:
class MLEStarSearcher:
    """Pipeline de recherche SOTA (partie de MLE-STAR)."""

    def __init__(self):
        self.llm = LLMClient()
        self.web_searcher = WebSearcher()
        self.extractor = ModelExtractor(self.llm)
        self.code_gen = SOTACodeGenerator(self.llm)

    def find_sota_models(self, task: str) -> Dict:
        """Trouve les modeles SOTA pour une tache donnee."""
        print(f"[WEB SEARCH] Recherche pour: {task}")

        # Search arXiv
        print("  - Recherche arXiv...")
        arxiv_results = self.web_searcher.search_arxiv(task)

        # Search Papers With Code (simule)
        print("  - Recherche Papers With Code...")
        pwc_results = self.web_searcher.search_papers_with_code(task)

        all_results = arxiv_results + pwc_results
        print(f"  - {len(all_results)} resultats trouves")

        # Extract models with LLM
        print("[EXTRACTOR] Extraction des modeles...")
        models = self.extractor.extract_models(all_results, task)
        print(f"  - {len(models)} modeles identifies")

        return {
            'task': task,
            'search_results': all_results,
            'models': models
        }

    def generate_baseline_code(self, task: str, models: List[ModelInfo]) -> str:
        """Genere du code de base."""
        print("[CODE GEN] Generation du code initial...")
        code = self.code_gen.generate_initial_code(task, models)
        return code
print("Classe MLEStarSearcher definie")

Classe MLEStarSearcher definie


## 7. Test du Pipeline

In [8]:
# Test avec une tache ML typique
searcher = MLEStarSearcher()

task = "image classification imagenet"
result = searcher.find_sota_models(task)

print("\n" + "="*50)
print("MODELES TROUVES:")
print("="*50)
for m in result['models']:
    print(f"\n- {m.name}")
    print(f"  Description: {m.description[:80]}...")
    print(f"  Paper: {m.paper_url}")

[WEB SEARCH] Recherche pour: image classification imagenet
  - Recherche arXiv...


  - Recherche Papers With Code...
  - 7 resultats trouves
[EXTRACTOR] Extraction des modeles...


  - 3 modeles identifies

MODELES TROUVES:

- Deep Learning Ensemble Models for Shoulder X-Ray Image Classification
  Description: This work focuses on classifying shoulder X-ray images to diagnose fractures usi...
  Paper: http://arxiv.org/abs/2102.00515v3

- Selective Synthetic Augmentation with HistoGAN for Histopathology Image Classification
  Description: The paper proposes a method using synthetic data augmentation with HistoGAN to i...
  Paper: http://arxiv.org/abs/2111.06399v1

- Technical Report for VIPriors Image Classification Challenge
  Description: This report details a submission to the VIPriors Image Classification Challenge,...
  Paper: http://arxiv.org/abs/2007.08722v1


## 8. Generation de Code

In [9]:
# Generer du code initial
if result['models']:
    code = searcher.generate_baseline_code(task, result['models'])
    print("\n" + "="*50)
    print("CODE GENERE:")
    print("="*50)
    print(code[:800] + "..." if len(code) > 800 else code)

[CODE GEN] Generation du code initial...



CODE GENERE:
# Exemple simple de script Python pour classification d'images ImageNet avec PyTorch
# Ce script charge un dataset ImageNet (ou un dataset similaire),
# prépare un modèle pré-entraîné (ResNet50),
# entraîne le modèle sur les données d'entraînement,
# et évalue la performance sur les données de validation.

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# 1. Chargement et préparation des données
# Ici on utilise des transformations classiques pour ImageNet
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # Moyenne ImageNet
             ...


## 9. Resume du Lab
### Ce que nous avons implemente
1. **WebSearcher**: Recherche sur arXiv et Papers With Code
2. **ModelExtractor**: Extraction des modeles via LLM
3. **SOTACodeGenerator**: Generation de code initial
### Limitations
- L'API Papers With Code est simulee (rate limiting)
- La recherche web reelle necessite une API (Tavily, Serper)
- Le code genere est un point de depart, pas une solution complete
### Integration MLE-STAR
Dans MLE-STAR complet, ce module est utilise pour:
1. Comprendre la competition Kaggle
2. Trouver les approches SOTA
3. Generer le code initial
4. Puis passer a l'ablation et raffinement
### Prochaine etape
- **Lab 14**: Ablation et Raffinement Cible

## Exercice

À vous de rechercher des modèles SOTA pour une autre tâche ML et de comparer les résultats !


In [10]:
# Exercice : Recherchez des modeles SOTA pour une autre tache
# Utilisez le MLEStarSearcher pour trouver des modeles pour "object detection"

# TODO: Definissez la tache de recherche
task_detection = "object detection coco"

print("=" * 50)
print(f"Recherche SOTA pour : {task_detection}")
print("=" * 50)

# TODO: Utilisez searcher.find_sota_models() pour trouver les modeles
# Indice: meme pattern que le test de la section 7
result_detection = None  # Remplacez None

# TODO: Affichez les modeles trouves
# Indice: parcourez result_detection['models'] et affichez name, description, code_url

# TODO: Comparez avec les resultats de classification d'images
# Indice: comparez len(result['models']) vs len(result_detection['models'])

# REFLEXION (repondez dans un commentaire):
# Quelle est la difference principale entre les modeles
# de classification d'images et ceux de detection d'objets ?
# ...


Recherche SOTA pour : object detection coco
